<a href="https://colab.research.google.com/github/muneer-ahmad10/Natural-Language-processing/blob/main/Day_13_mini_transformer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import torch
import torch.nn as nn
import math

In [3]:
embeddings=nn.Embedding(3,4)

In [4]:
tokens=torch.tensor([0,1,2])

In [5]:
x=embeddings(tokens)

In [6]:
x.shape

torch.Size([3, 4])

In [7]:
x

tensor([[-2.0330,  0.8625, -0.9718, -0.7669],
        [-2.3534, -0.7663, -0.1943, -0.3759],
        [-0.4040,  1.1153,  1.6384,  0.6528]], grad_fn=<EmbeddingBackward0>)

In [8]:
Wq=nn.Linear(4,4)
Wk=nn.Linear(4,4)
Wv=nn.Linear(4,4)

In [9]:
Wq

Linear(in_features=4, out_features=4, bias=True)

In [10]:
Wk

Linear(in_features=4, out_features=4, bias=True)

In [11]:
Q=Wq(x)
K=Wk(x)
V=Wv(x)

In [12]:
Q

tensor([[ 0.6879, -0.5529, -1.5155, -1.1625],
        [ 1.1862, -0.0397, -0.7559, -1.0345],
        [-0.0322, -0.8242, -0.3167, -0.1026]], grad_fn=<AddmmBackward0>)

In [13]:
K

tensor([[-0.0223, -0.3521,  0.2466,  1.5505],
        [ 0.5365, -0.7792,  0.3564,  0.5599],
        [-0.3602, -0.5839, -0.3109,  0.0597]], grad_fn=<AddmmBackward0>)

In [14]:
V

tensor([[ 0.3519,  0.5651,  0.2166,  1.2868],
        [ 0.9667,  1.0299, -0.7089,  0.6940],
        [-0.2552, -0.1643, -0.0155,  0.8607]], grad_fn=<AddmmBackward0>)

In [15]:
scores=Q @ K.T

In [16]:
scores=scores / math.sqrt(4)

In [17]:
weights=torch.softmax(
    scores,
    dim=-1
)

In [18]:
context=weights @ V

In [19]:
print(context)

tensor([[ 0.2442,  0.3441, -0.2125,  0.8688],
        [ 0.3612,  0.4631, -0.2594,  0.8701],
        [ 0.3404,  0.4563, -0.1899,  0.9238]], grad_fn=<MmBackward0>)


**Explain self-attention**

Self-attention allows each token to compare itself with every other token using Query and Key vectors. The resulting attention weights determine how much information to gather from the Value vectors, creating a context-aware representation of each word

# **Day_14 _Multi-Head Attention**

where 8 attention mechanisms run simultaneously, and you'll finally understand why GPT uses many heads instead of one. This is the point where our tiny Transformer starts looking like a real GPT

In [20]:
embed_dim=8
num_heads=2

In [21]:
head_dim = embed_dim // num_heads

print(head_dim)

4


**Real GPT**

GPT-2:

12 heads

GPT-3:

96 heads

Modern models:

100+

In [22]:
import torch
import torch.nn as nn
import math

In [23]:
class MultiHeadAttention(nn.Module):

    def __init__(self, embed_dim, num_heads):
        super().__init__()

        assert embed_dim % num_heads == 0, \
            "Embedding dimension must be divisible by number of heads"

        self.embed_dim = embed_dim
        self.num_heads = num_heads
        self.head_dim = embed_dim // num_heads

        # Query, Key, Value projections
        self.Wq = nn.Linear(embed_dim, embed_dim)
        self.Wk = nn.Linear(embed_dim, embed_dim)
        self.Wv = nn.Linear(embed_dim, embed_dim)

        # Final output projection
        self.fc_out = nn.Linear(embed_dim, embed_dim)

    def forward(self, x):

        batch_size = x.shape[0]
        seq_len = x.shape[1]

        # Generate Q, K, V
        Q = self.Wq(x)
        K = self.Wk(x)
        V = self.Wv(x)

        # Split into multiple heads
        Q = Q.view(
            batch_size,
            seq_len,
            self.num_heads,
            self.head_dim
        )

        K = K.view(
            batch_size,
            seq_len,
            self.num_heads,
            self.head_dim
        )

        V = V.view(
            batch_size,
            seq_len,
            self.num_heads,
            self.head_dim
        )

        # Move head dimension forward
        Q = Q.transpose(1, 2)
        K = K.transpose(1, 2)
        V = V.transpose(1, 2)

        # Attention scores
        scores = torch.matmul(
            Q,
            K.transpose(-2, -1)
        )

        # Scale
        scores = scores / math.sqrt(self.head_dim)

        # Softmax
        attention_weights = torch.softmax(
            scores,
            dim=-1
        )

        # Context vectors
        context = torch.matmul(
            attention_weights,
            V
        )

        # Restore dimensions
        context = context.transpose(1, 2)

        context = context.reshape(
            batch_size,
            seq_len,
            self.embed_dim
        )

        # Final projection
        output = self.fc_out(context)

        return output

In [24]:
batch_size = 1
seq_len = 3
embed_dim = 8
num_heads = 2

# Example input
x = torch.randn(
    batch_size,
    seq_len,
    embed_dim
)

mha = MultiHeadAttention(
    embed_dim=embed_dim,
    num_heads=num_heads
)

output = mha(x)

print("Input Shape :", x.shape)
print("Output Shape:", output.shape)

print("\nOutput Tensor:\n")
print(output)

Input Shape : torch.Size([1, 3, 8])
Output Shape: torch.Size([1, 3, 8])

Output Tensor:

tensor([[[-0.8889,  0.3318,  0.1819,  0.2544,  0.1622,  0.0293,  0.4517,
           0.0852],
         [-0.8799,  0.3912,  0.1631,  0.2153,  0.1049,  0.0630,  0.4318,
           0.0941],
         [-0.9835,  0.3432,  0.1800,  0.2038,  0.1838,  0.0377,  0.5234,
           0.1031]]], grad_fn=<ViewBackward0>)


**Why do we split embeddings into heads?**

Different attention heads learn different relationships in the sentence. One head may learn grammar, another semantic meaning, another long-range dependencies. Their outputs are combined to create richer contextual representations.

# Transformer Block

In [34]:
class Transformer(nn.Module):
  def __init__(self, embed_dim,num_heads):
      super().__init__()

      self.attention=MultiHeadAttention(
          embed_dim,
          num_heads
      )

      self.norm1=nn.LayerNorm(embed_dim)

      self.ffn=nn.Sequential(
          nn.Linear(embed_dim,embed_dim*4),
          nn.ReLU(),
          nn.Linear(embed_dim*4,embed_dim)
      )

      self.norm2=nn.LayerNorm(embed_dim)

  def forward(self,x):
    attention_output=self.attention(x)

    #residuaal + Norm

    x=self.norm1(x + attention_output)

    ffn_output=self.ffn(x)

    x=self.norm2(x+ffn_output)

    return x

In [35]:
x = torch.randn(
    1,
    3,
    8
)

block = Transformer(
    embed_dim=8,
    num_heads=2
)

output = block(x)

print(output.shape)

torch.Size([1, 3, 8])
